## Clustering: Funciones

Utilidades compartidas por `a_preparacion`, `b_modelos` y `c_interpretacion` (`%run funciones.ipynb`)

### Imports

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

### Rutas

- `clust_dir`: intermedios de esta parte (`data/clustering/`, fuera de git)
- El artefacto final para la web va a `data/artefactos/clustering_proveedores`

In [ ]:
clust_dir = proyecto / "data" / "clustering"
clust_dir.mkdir(parents=True, exist_ok=True)
SEMILLA = 42

### Codo de una curva

- Punto de máxima distancia a la cuerda entre el primer y el último punto (método *kneedle* simplificado)
- Se usa para fijar `eps` de DBSCAN sin elegirlo a ojo

In [ ]:
def codo(y):
    y = np.asarray(y, dtype=float)
    x = np.linspace(0, 1, len(y))
    yn = (y - y.min()) / (y.max() - y.min())
    return int(np.argmax(np.abs(yn - x)))  # curva creciente: distancia a la diagonal

### Perfiles y privacidad

- `perfil`: mediana de las variables originales por grupo, más tamaño y participación
- `publicable`: personas naturales (`PN:`) sin nombre ni documento (su clave contiene el DNI)

In [ ]:
def perfil(df, grupo, cols):
    p = df.groupby(grupo)[cols].median()
    p.insert(0, "pct", df[grupo].value_counts(normalize=True).reindex(p.index).mul(100).round(1))
    p.insert(0, "n", df[grupo].value_counts().reindex(p.index))
    return p

def publicable(df):
    pn = df.proveedor_key.str.startswith("PN:")
    return df.assign(nombre=df.nombre.where(~pn, "PERSONA NATURAL"),
                     ruc=df.proveedor_key.str.extract(r"^RUC:(\d{11})$")[0],
                     proveedor_key=df.proveedor_key.where(~pn, "PN:" + df.index.astype(str)))